# PortPy Tutorial: Full Function Reference

A working debug notebook that exercises **every public function** in every `portpy.metrics`
submodule, plus the `portpy.core` and `portpy.explain` helpers.

**Data**: real market data pulled live from **Alpaca** (stocks, ETFs, commodity ETFs, crypto)
and a risk-free rate pulled live from the **Federal Reserve (FRED)**.

**Portfolios**: three independent portfolios are built, each spanning all five asset
categories (stocks, ETFs, crypto, commodities, a Fed-rate-derived cash leg) and each
carrying at least one **negative (short) weight**, so every metric below is exercised on
long-only and long/short books:

| Portfolio | Style | Shorts |
|---|---|---|
| `growth` | US growth stocks + crypto | short `XOM` (energy hedge) |
| `balanced` | Broad multi-asset diversification | short `NVDA` (valuation hedge) |
| `macro` | Crypto-heavy long/short macro | short `USO`, short `SPY` |

Sections below map 1:1 to PortPy's `metrics/` submodules: returns, risk, performance,
drawdowns, rolling, distributions, benchmarks, covariance, summary, costs -
followed by the explainability layer and the `Portfolio.metrics` auto-fill mechanics.

`portpy.models`, `portpy.strategies`, and `portpy.visualization` are not yet implemented in
this PortPy version (0.1.0), so they're out of scope here.


## 0. Setup


In [1]:
from __future__ import annotations

import os
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from alpaca.data.historical import CryptoHistoricalDataClient, StockHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest, StockBarsRequest
from alpaca.data.timeframe import TimeFrame

import portpy
from portpy import Portfolio
from portpy.core import (
    AssetClass,
    align_calendars,
    calendar_coverage_report,
    convert_to_base_currency,
    detect_frequency,
    equal_weights,
    normalize_weights,
)
from portpy.explain import available, get

load_dotenv()
ALPACA_KEY = os.getenv("ALPACA_KEY")
ALPACA_SECRET = os.getenv("ALPACA_SECRET")
assert ALPACA_KEY and ALPACA_SECRET, "Set ALPACA_KEY / ALPACA_SECRET in a .env file."
print("PortPy version:", portpy.__version__)
print("Alpaca credentials loaded:", bool(ALPACA_KEY) and bool(ALPACA_SECRET))


PortPy version: 0.2.0
Alpaca credentials loaded: True


## 1. Data Acquisition -- Alpaca (stocks, ETFs, commodities, crypto) + FRED (risk-free rate)

Universe:
- **Stocks**: AAPL, MSFT, NVDA, JPM, XOM
- **ETFs**: SPY, QQQ, IWM
- **Commodities** (via liquid ETF proxies, fetched through the same Alpaca stock client): GLD (gold), SLV (silver), USO (oil)
- **Crypto** (Alpaca, 24/7): BTC/USD, ETH/USD, SOL/USD
- **Risk-free**: 3-Month Treasury Bill yield (`DGS3MO`) straight from the Fed's FRED database


In [2]:
LOOKBACK_DAYS = 4 * 365

STOCKS = ["AAPL", "MSFT", "NVDA", "JPM", "XOM"]
ETFS = ["SPY", "QQQ", "IWM"]
COMMODITIES = ["GLD", "SLV", "USO"]
CRYPTO = ["BTC/USD", "ETH/USD", "SOL/USD"]

stock_client = StockHistoricalDataClient(ALPACA_KEY, ALPACA_SECRET)
equities_request = StockBarsRequest(
    symbol_or_symbols=STOCKS + ETFS + COMMODITIES,
    timeframe=TimeFrame.Day,
    start=datetime.now() - timedelta(days=LOOKBACK_DAYS),
)
equities_bars = stock_client.get_stock_bars(equities_request)
raw_equities = equities_bars.df["close"].unstack(level="symbol")
raw_equities.index = raw_equities.index.tz_localize(None).normalize()
print(f"Fetched {raw_equities.shape[1]} stock/ETF/commodity symbols, {raw_equities.shape[0]} rows.")
raw_equities.tail()


Fetched 11 stock/ETF/commodity symbols, 1001 rows.


symbol,AAPL,GLD,IWM,JPM,MSFT,NVDA,QQQ,SLV,SPY,USO,XOM
timestamp,,,,,,,,,,,
2026-09-08,316.2200,399.7200,294.6700,353.5100,493.9500,225.7300,718.3600,59.3700,765.9600,146.0300,160.6600
2026-09-09,315.3400,403.3500,290.6400,354.7100,491.6500,223.6700,716.3100,60.7200,762.4000,149.9700,164.2300
2026-09-10,326.5700,396.3600,287.7000,353.5600,492.4400,218.3600,708.6900,57.5000,757.8300,158.3800,165.2300
2026-09-11,332.2700,398.7700,288.8900,356.2300,495.6300,218.2900,714.8800,58.1200,764.2900,154.9000,165.9900
2026-09-14,333.0800,392.8400,287.9100,350.1300,505.4100,210.9600,709.1800,56.8400,760.8800,156.6600,165.0800


In [3]:
crypto_client = CryptoHistoricalDataClient()
crypto_request = CryptoBarsRequest(
    symbol_or_symbols=CRYPTO,
    timeframe=TimeFrame.Day,
    start=datetime.now() - timedelta(days=LOOKBACK_DAYS),
)
crypto_bars = crypto_client.get_crypto_bars(crypto_request)
raw_crypto = crypto_bars.df["close"].unstack(level="symbol")
raw_crypto.index = raw_crypto.index.tz_localize(None).normalize()
raw_crypto.columns = [c.replace("/USD", "") for c in raw_crypto.columns]
print(f"Fetched {raw_crypto.shape[1]} crypto symbols, {raw_crypto.shape[0]} rows.")
raw_crypto.tail()


Fetched 3 crypto symbols, 1460 rows.


,BTC,ETH,SOL
timestamp,,,
2026-09-10,"76,535.8750","2,436.8075",98.6025
2026-09-11,"77,212.8315","2,516.0660",102.4335
2026-09-12,"77,266.3850","2,525.0450",101.7655
2026-09-13,"76,805.2780","2,476.5845",99.2331
2026-09-14,"78,173.6460","2,513.7650",102.4840


In [4]:
# The Fed publishes this on FRED; the plain CSV export needs no API key.
def fetch_fred_series(series_id: str, start: str, end: str) -> pd.Series:
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd={start}&coed={end}"
    raw = pd.read_csv(url, index_col=0, parse_dates=True)
    return raw[series_id].rename(series_id)

fred_start = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")
fred_end = datetime.now().strftime("%Y-%m-%d")

t3m_yield_pct = fetch_fred_series("DGS3MO", fred_start, fred_end)  # 3-Month Treasury, Secondary Market (Fed H.15)
fed_funds_pct = fetch_fred_series("FEDFUNDS", fred_start, fred_end)  # Effective Federal Funds Rate (monthly)

risk_free_annual = (t3m_yield_pct / 100.0).dropna()
print(f"Fetched {len(risk_free_annual)} daily 3-Month T-Bill observations from FRED.")
print(
    f"Latest 3M T-Bill yield: {risk_free_annual.iloc[-1]:.2%} annual  |  "
    f"Latest Fed Funds rate: {fed_funds_pct.dropna().iloc[-1] / 100:.2%} annual"
)
risk_free_annual.tail()


Fetched 997 daily 3-Month T-Bill observations from FRED.
Latest 3M T-Bill yield: 4.07% annual  |  Latest Fed Funds rate: 3.63% annual


observation_date
2026-09-04   0.0391
2026-09-08   0.0394
2026-09-09   0.0395
2026-09-10   0.0400
2026-09-11   0.0407
Name: DGS3MO, dtype: float64

## 2. Calendar Alignment (`portpy.core`) -- combining a 24/7 crypto calendar with Mon-Fri equities

PortPy never aligns data for you. `detect_frequency`, `calendar_coverage_report`, and
`align_calendars` are the tools you call *before* constructing a `Portfolio`.


In [5]:
combined_raw = pd.concat([raw_equities, raw_crypto], axis=1, sort=True)
print(f"detect_frequency: equities -> {detect_frequency(raw_equities.index)!r}, crypto -> {detect_frequency(raw_crypto.index)!r}")
calendar_coverage_report(combined_raw)


detect_frequency: equities -> 'daily-5', crypto -> 'daily-7'


,frequency,first_date,last_date,n_observations,n_missing_in_frame
asset,,,,,
AAPL,daily-5,2022-09-16,2026-09-14,1001,459
GLD,daily-5,2022-09-16,2026-09-14,1001,459
IWM,daily-5,2022-09-16,2026-09-14,1001,459
JPM,daily-5,2022-09-16,2026-09-14,1001,459
MSFT,daily-5,2022-09-16,2026-09-14,1001,459
NVDA,daily-5,2022-09-16,2026-09-14,1001,459
QQQ,daily-5,2022-09-16,2026-09-14,1001,459
SLV,daily-5,2022-09-16,2026-09-14,1001,459
SPY,daily-5,2022-09-16,2026-09-14,1001,459


In [6]:
n_before = len(combined_raw)
aligned_intersection = align_calendars(combined_raw, method="intersection")
aligned_union = align_calendars(combined_raw, method="ffill_union")
aligned_bdays = align_calendars(combined_raw, method="business_days")

print(f"Before alignment:  {n_before} rows, {int(combined_raw.isna().any(axis=1).sum())} rows with >=1 NaN.")
print(f"'intersection'  -> {len(aligned_intersection)} rows (equity-only calendar; crypto weekend moves dropped)")
print(f"'ffill_union'   -> {len(aligned_union)} rows (every day crypto trades; weekend equity prices carried forward)")
print(f"'business_days' -> {len(aligned_bdays)} rows (Mon-Fri only; everything forward-filled onto it)")
assert aligned_union.isna().sum().sum() == 0


Before alignment:  1460 rows, 745 rows with >=1 NaN.
'intersection'  -> 715 rows (equity-only calendar; crypto weekend moves dropped)
'ffill_union'   -> 1460 rows (every day crypto trades; weekend equity prices carried forward)
'business_days' -> 1042 rows (Mon-Fri only; everything forward-filled onto it)


## 3. Building a Synthetic CASH Leg from the Fed's Risk-Free Rate

Every metric function needing an `rf` accepts a plain float, but a *portfolio holding* has to
be a price series. This turns the Fed's daily T-Bill yield into a compounding NAV-style price
column so "risk-free" can sit in the price DataFrame right alongside stocks/ETFs/crypto/commodities.


In [7]:
def synthetic_cash_price(annual_rate: pd.Series, target_index: pd.DatetimeIndex, periods_per_year: int = 365, base: float = 100.0) -> pd.Series:
    rate_on_calendar = annual_rate.reindex(target_index).ffill().bfill()
    daily_rate = (1.0 + rate_on_calendar) ** (1.0 / periods_per_year) - 1.0
    return (base * (1.0 + daily_rate).cumprod()).rename("CASH")

cash_price = synthetic_cash_price(risk_free_annual, aligned_union.index)

master = aligned_union.copy()
master["CASH"] = cash_price
print(f"Master frame: {master.shape[1]} assets x {master.shape[0]} days ({master.index.min().date()} -> {master.index.max().date()})")
master.tail()


Master frame: 15 assets x 1460 days (2022-09-16 -> 2026-09-14)


,AAPL,GLD,IWM,JPM,MSFT,NVDA,QQQ,SLV,SPY,USO,XOM,BTC,ETH,SOL,CASH
timestamp,,,,,,,,,,,,,,,
2026-09-10,326.5700,396.3600,287.7000,353.5600,492.4400,218.3600,708.6900,57.5000,757.8300,158.3800,165.2300,"76,535.8750","2,436.8075",98.6025,119.7659
2026-09-11,332.2700,398.7700,288.8900,356.2300,495.6300,218.2900,714.8800,58.1200,764.2900,154.9000,165.9900,"77,212.8315","2,516.0660",102.4335,119.7790
2026-09-12,332.2700,398.7700,288.8900,356.2300,495.6300,218.2900,714.8800,58.1200,764.2900,154.9000,165.9900,"77,266.3850","2,525.0450",101.7655,119.7921
2026-09-13,332.2700,398.7700,288.8900,356.2300,495.6300,218.2900,714.8800,58.1200,764.2900,154.9000,165.9900,"76,805.2780","2,476.5845",99.2331,119.8051
2026-09-14,333.0800,392.8400,287.9100,350.1300,505.4100,210.9600,709.1800,56.8400,760.8800,156.6600,165.0800,"78,173.6460","2,513.7650",102.4840,119.8182


## 4. Building Three Portfolios (`portpy.core.weights`, `portpy.Portfolio`)

Each portfolio deliberately carries at least one **negative weight** (a short position) to make
sure every metric below is exercised on a long/short book, not just a long-only one.


In [8]:
weights_growth = {
    "AAPL": 0.22, "MSFT": 0.18, "NVDA": 0.18,    # US mega-cap tech (stocks)
    "QQQ": 0.12,                                 # Nasdaq-100 ETF
    "BTC": 0.14, "ETH": 0.08,                    # Crypto
    "GLD": 0.08,                                 # Commodity (gold)
    "CASH": 0.05,                                # Fed risk-free leg
    "XOM": -0.15,                                # SHORT: energy hedge against the growth book
}

weights_balanced = {
    "SPY": 0.20, "IWM": 0.10,                    # Broad + small-cap equity ETFs
    "JPM": 0.12,                                 # Stock (financials)
    "GLD": 0.12, "SLV": 0.08, "USO": 0.08,       # Commodities
    "BTC": 0.10, "ETH": 0.07,                    # Crypto
    "CASH": 0.15,                                # Fed risk-free leg
    "NVDA": -0.10,                               # SHORT: valuation hedge on AI momentum
}

weights_macro = {
    "MSFT": 0.10,                                # Stock anchor
    "QQQ": 0.15,                                 # Long tech ETF
    "BTC": 0.25, "ETH": 0.15, "SOL": 0.10,       # Crypto-heavy book
    "GLD": 0.15,                                 # Long commodity
    "USO": -0.10,                                # SHORT: macro bet against energy
    "SPY": -0.15,                                # SHORT: net market hedge
    "CASH": 0.20,                                # Fed risk-free ballast
}

asset_class_map = {
    "AAPL": AssetClass.EQUITY, "MSFT": AssetClass.EQUITY, "NVDA": AssetClass.EQUITY,
    "JPM": AssetClass.EQUITY, "XOM": AssetClass.EQUITY,
    "SPY": AssetClass.EQUITY, "QQQ": AssetClass.EQUITY, "IWM": AssetClass.EQUITY,
    "GLD": AssetClass.COMMODITY, "SLV": AssetClass.COMMODITY, "USO": AssetClass.COMMODITY,
    "BTC": AssetClass.CRYPTO, "ETH": AssetClass.CRYPTO, "SOL": AssetClass.CRYPTO,
    "CASH": AssetClass.BOND,
}

for label, w in [("growth", weights_growth), ("balanced", weights_balanced), ("macro", weights_macro)]:
    gross = sum(abs(v) for v in w.values())
    net = sum(w.values())
    shorts = {k: v for k, v in w.items() if v < 0}
    print(f"{label:10s}: {len(w)} assets | gross exposure {gross:.2f} | net exposure {net:.2f} | shorts: {shorts}")


growth    : 9 assets | gross exposure 1.20 | net exposure 0.90 | shorts: {'XOM': -0.15}
balanced  : 10 assets | gross exposure 1.12 | net exposure 0.92 | shorts: {'NVDA': -0.1}
macro     : 9 assets | gross exposure 1.35 | net exposure 0.85 | shorts: {'USO': -0.1, 'SPY': -0.15}


### 4.1 `portpy.core.weights` -- `equal_weights` and `normalize_weights` on a long/short book


In [9]:
print("equal_weights on 4 assets:")
print(equal_weights(["A", "B", "C", "D"]))

print("\nnormalize_weights rescales an arbitrary long/short vector to sum to 1 (default allow_negative=True):")
print(normalize_weights(weights_macro))

print("\nnormalize_weights with allow_negative=False raises on a book with shorts (e.g. for a long-only optimizer):")
try:
    normalize_weights(weights_macro, allow_negative=False)
except ValueError as exc:
    print(f"  ValueError: {exc}")


equal_weights on 4 assets:
A   0.2500
B   0.2500
C   0.2500
D   0.2500
Name: weight, dtype: float64

normalize_weights rescales an arbitrary long/short vector to sum to 1 (default allow_negative=True):
MSFT    0.1176
QQQ     0.1765
BTC     0.2941
ETH     0.1765
SOL     0.1176
GLD     0.1765
USO    -0.1176
SPY    -0.1765
CASH    0.2353
Name: weight, dtype: float64

normalize_weights with allow_negative=False raises on a book with shorts (e.g. for a long-only optimizer):
  ValueError: Negative weights are not allowed here (long-only constraint).


In [10]:
def build_portfolio(name: str, weights: dict, frequency: int = 365) -> Portfolio:
    cols = list(weights.keys())
    classes = {k: asset_class_map[k] for k in cols}
    return Portfolio(
        master[cols],
        weights=weights,
        name=name,
        frequency=frequency,
        risk_free_rate=risk_free_annual.iloc[-1],
        asset_classes=classes,
    )

p_growth = build_portfolio("Growth & Crypto Long/Short", weights_growth)
p_balanced = build_portfolio("Diversified Multi-Asset Balanced", weights_balanced)
p_macro = build_portfolio("Long/Short Macro", weights_macro)

portfolios = {"growth": p_growth, "balanced": p_balanced, "macro": p_macro}
for p in portfolios.values():
    print(p)
    print("  weights:", p.weights.round(4).to_dict())
    print("  asset_classes:", p.asset_classes)


Portfolio(name='Growth & Crypto Long/Short', assets=9, n_obs=1460, frequency=365)
  weights: {'AAPL': 0.2444, 'MSFT': 0.2, 'NVDA': 0.2, 'QQQ': 0.1333, 'BTC': 0.1556, 'ETH': 0.0889, 'GLD': 0.0889, 'CASH': 0.0556, 'XOM': -0.1667}
  asset_classes: {'AAPL': <AssetClass.EQUITY: 'equity'>, 'MSFT': <AssetClass.EQUITY: 'equity'>, 'NVDA': <AssetClass.EQUITY: 'equity'>, 'QQQ': <AssetClass.EQUITY: 'equity'>, 'BTC': <AssetClass.CRYPTO: 'crypto'>, 'ETH': <AssetClass.CRYPTO: 'crypto'>, 'GLD': <AssetClass.COMMODITY: 'commodity'>, 'CASH': <AssetClass.BOND: 'bond'>, 'XOM': <AssetClass.EQUITY: 'equity'>}
Portfolio(name='Diversified Multi-Asset Balanced', assets=10, n_obs=1460, frequency=365)
  weights: {'SPY': 0.2174, 'IWM': 0.1087, 'JPM': 0.1304, 'GLD': 0.1304, 'SLV': 0.087, 'USO': 0.087, 'BTC': 0.1087, 'ETH': 0.0761, 'CASH': 0.163, 'NVDA': -0.1087}
  asset_classes: {'SPY': <AssetClass.EQUITY: 'equity'>, 'IWM': <AssetClass.EQUITY: 'equity'>, 'JPM': <AssetClass.EQUITY: 'equity'>, 'GLD': <AssetClass.COMM

In [11]:
# A shared benchmark for every alpha/beta/capture-ratio style metric below.
benchmark_returns = master["SPY"].pct_change().dropna().rename("SPY")
print(f"Benchmark: SPY, {len(benchmark_returns)} daily returns, {benchmark_returns.index.min().date()} -> {benchmark_returns.index.max().date()}")


Benchmark: SPY, 1459 daily returns, 2022-09-17 -> 2026-09-14


### 4.2 Bonus: `convert_to_base_currency` (FX conversion, self-contained example)


In [12]:
demo_prices = pd.DataFrame({"EUR_STOCK": [100.0, 101.0, 102.5]}, index=pd.date_range("2026-01-01", periods=3))
demo_fx = pd.DataFrame({"EUR": [1.08, 1.09, 1.10]}, index=pd.date_range("2026-01-01", periods=3))
demo_usd = convert_to_base_currency(demo_prices, demo_fx, asset_currencies={"EUR_STOCK": "EUR"}, base_currency="USD")
demo_usd


,EUR_STOCK
2026-01-01,108.0000
2026-01-02,110.0900
2026-01-03,112.7500


### 4.3 Bonus: `calculate_fx_spread` (deriving a real fx_spread_bps from bid/ask FX quotes)


In [13]:
from portpy.core.currency import calculate_fx_spread

demo_bid = pd.DataFrame({"EUR": [1.070, 1.080, 1.090]}, index=pd.date_range("2026-01-01", periods=3))
demo_ask = pd.DataFrame({"EUR": [1.072, 1.082, 1.093]}, index=pd.date_range("2026-01-01", periods=3))
eur_spread_fraction = calculate_fx_spread(demo_bid, demo_ask)
print(eur_spread_fraction)
# Feed straight into fx_spread_costs as fx_spread_bps after multiplying by 10,000:
print(f"\nmean EUR fx_spread_bps: {eur_spread_fraction['EUR'].mean() * 10_000:.2f} bps")

              EUR
2026-01-01 0.0019
2026-01-02 0.0019
2026-01-03 0.0027

mean EUR fx_spread_bps: 21.55 bps


## 5. Portfolio Core Accessors & Methods


In [14]:
for label, p in portfolios.items():
    print(f"--- {label} ({p.name}) ---")
    print("asset_names:", p.asset_names)
    print("num_assets:", p.num_assets)
    print("prices.shape:", p.prices.shape)
    print("weights:")
    print(p.weights)
    print("asset_returns().tail(2):")
    print(p.asset_returns().tail(2))
    print("returns().tail(2):")
    print(p.returns().tail(2))
    print("returns(log=True).tail(2):")
    print(p.returns(log=True).tail(2))
    print("returns(period=21).tail(2)  (~monthly blocks):")
    print(p.returns(period=21).tail(2))
    print("price_index().tail(2):")
    print(p.price_index().tail(2))
    print()


--- growth (Growth & Crypto Long/Short) ---
asset_names: ['AAPL', 'MSFT', 'NVDA', 'QQQ', 'BTC', 'ETH', 'GLD', 'CASH', 'XOM']
num_assets: 9
prices.shape: (1460, 9)
weights:
AAPL    0.2444
MSFT    0.2000
NVDA    0.2000
QQQ     0.1333
BTC     0.1556
ETH     0.0889
GLD     0.0889
CASH    0.0556
XOM    -0.1667
Name: weight, dtype: float64
asset_returns().tail(2):
             AAPL   MSFT    NVDA     QQQ     BTC     ETH     GLD   CASH     XOM
timestamp                                                                      
2026-09-13 0.0000 0.0000  0.0000  0.0000 -0.0060 -0.0192  0.0000 0.0001  0.0000
2026-09-14 0.0024 0.0197 -0.0336 -0.0080  0.0178  0.0150 -0.0149 0.0001 -0.0055
returns().tail(2):
timestamp
2026-09-13   -0.0026
2026-09-14    0.0005
Name: Growth & Crypto Long/Short, dtype: float64
returns(log=True).tail(2):
timestamp
2026-09-13   -0.0026
2026-09-14    0.0005
Name: Growth & Crypto Long/Short, dtype: float64
returns(period=21).tail(2)  (~monthly blocks):
2026-09-04    0.0842
202

In [15]:
demo_portfolio = build_portfolio("Demo (mutation sandbox)", weights_growth)
print("original weights:", demo_portfolio.weights.round(3).to_dict())
demo_portfolio.set_weights({k: 1.0 for k in demo_portfolio.asset_names})  # unnormalized -> normalize_weights rescales it
print("after set_weights (equal, pre-normalize):", demo_portfolio.weights.round(3).to_dict())

print("original risk_free_rate:", demo_portfolio.risk_free_rate)
demo_portfolio.set_risk_free_rate(0.02)
print("after set_risk_free_rate(0.02):", demo_portfolio.risk_free_rate)
print(repr(demo_portfolio))


original weights: {'AAPL': 0.244, 'MSFT': 0.2, 'NVDA': 0.2, 'QQQ': 0.133, 'BTC': 0.156, 'ETH': 0.089, 'GLD': 0.089, 'CASH': 0.056, 'XOM': -0.167}
after set_weights (equal, pre-normalize): {'AAPL': 0.111, 'MSFT': 0.111, 'NVDA': 0.111, 'QQQ': 0.111, 'BTC': 0.111, 'ETH': 0.111, 'GLD': 0.111, 'CASH': 0.111, 'XOM': 0.111}
original risk_free_rate: 0.0407
after set_risk_free_rate(0.02): 0.02
Portfolio(name='Demo (mutation sandbox)', assets=9, n_obs=1460, frequency=365)


## 6. Metrics -- Returns (`portpy.metrics.returns`)


In [16]:
from portpy.metrics.returns import (
    active_returns,
    annualized_return,
    average_return,
    cagr,
    cumulative_returns,
    excess_returns,
    log_returns,
    prices_from_returns,
    rebased_returns,
    simple_returns,
    total_return,
)

for label, p in portfolios.items():
    sr = simple_returns(p.prices)
    lr = log_returns(p.prices)
    print(f"{label}: simple_returns shape={sr.shape}, log_returns shape={lr.shape}")
    print(sr.iloc[-1].round(4).to_dict())


growth: simple_returns shape=(1459, 9), log_returns shape=(1459, 9)
{'AAPL': 0.0024, 'MSFT': 0.0197, 'NVDA': -0.0336, 'QQQ': -0.008, 'BTC': 0.0178, 'ETH': 0.015, 'GLD': -0.0149, 'CASH': 0.0001, 'XOM': -0.0055}
balanced: simple_returns shape=(1459, 10), log_returns shape=(1459, 10)
{'SPY': -0.0045, 'IWM': -0.0034, 'JPM': -0.0171, 'GLD': -0.0149, 'SLV': -0.022, 'USO': 0.0114, 'BTC': 0.0178, 'ETH': 0.015, 'CASH': 0.0001, 'NVDA': -0.0336}
macro: simple_returns shape=(1459, 9), log_returns shape=(1459, 9)
{'MSFT': 0.0197, 'QQQ': -0.008, 'BTC': 0.0178, 'ETH': 0.015, 'SOL': 0.0328, 'GLD': -0.0149, 'USO': 0.0114, 'SPY': -0.0045, 'CASH': 0.0001}


In [17]:
for label, p in portfolios.items():
    r = p.returns()
    cum = cumulative_returns(r)
    reconstructed_prices = prices_from_returns(r, base=100.0)
    roundtrip_ok = np.allclose(simple_returns(reconstructed_prices).to_numpy(), r.to_numpy())
    print(f"{label}: total cumulative return={cum.iloc[-1]:+.2%}, prices_from_returns roundtrips exactly: {roundtrip_ok}")


growth: total cumulative return=+242.26%, prices_from_returns roundtrips exactly: True


balanced: total cumulative return=+112.38%, prices_from_returns roundtrips exactly: True
macro: total cumulative return=+315.13%, prices_from_returns roundtrips exactly: True


In [18]:
for label, p in portfolios.items():
    rebased = rebased_returns(p.price_index(), base=100.0)
    print(f"{label}: rebased_returns -> starts at {rebased.iloc[0]:.2f}, ends at {rebased.iloc[-1]:.2f}")


growth: rebased_returns -> starts at 100.00, ends at 342.26
balanced: rebased_returns -> starts at 100.00, ends at 212.38
macro: rebased_returns -> starts at 100.00, ends at 415.13


In [19]:
for label, p in portfolios.items():
    tr = p.metrics.total_return()
    tr_result = p.metrics.total_return(as_result=True)
    print(f"{label}: total_return={tr:+.2%}  |  as_result -> {tr_result!r}")
tr_result.explain()


growth: total_return=+242.26%  |  as_result -> total_return=2.4226 % - +242.3% total return
balanced: total_return=+112.38%  |  as_result -> total_return=1.12377 % - +112.4% total return
macro: total_return=+315.13%  |  as_result -> total_return=3.15131 % - +315.1% total return
total_return (metric)

What it is:
  The total compounded gain or loss achieved over the entire investment period without converting it into an annual rate.

Formula:
  prod(1 + r_t) - 1

How to read it:
  A value of 0.35 means an investment gained 35% from start to finish.

Good vs. bad:
  Useful for measuring absolute performance, but comparisons should use the same time period and similar risk exposure.

Caveats:
  Not annualized. The same total return can represent very different performance depending on whether it occurred over months or years.

This result:
  +315.1% total return


'total_return (metric)\n=====================\n\nWhat it is:\n  The total compounded gain or loss achieved over the entire investment period without converting it into an annual rate.\n\nFormula:\n  prod(1 + r_t) - 1\n\nHow to read it:\n  A value of 0.35 means an investment gained 35% from start to finish.\n\nGood vs. bad:\n  Useful for measuring absolute performance, but comparisons should use the same time period and similar risk exposure.\n\nCaveats:\n  Not annualized. The same total return can represent very different performance depending on whether it occurred over months or years.\n\nThis result:\n  +315.1% total return'

In [20]:
for label, p in portfolios.items():
    geo = p.metrics.annualized_return(geometric=True)
    arith = p.metrics.annualized_return(geometric=False)
    print(f"{label}: annualized_return geometric={geo:+.2%}, arithmetic={arith:+.2%}")


growth: annualized_return geometric=+36.04%, arithmetic=+34.85%
balanced: annualized_return geometric=+20.73%, arithmetic=+20.19%
macro: annualized_return geometric=+42.78%, arithmetic=+46.19%


In [21]:
for label, p in portfolios.items():
    c = p.metrics.cagr(as_result=True)
    print(f"{label}: {c!r}")


growth: cagr=0.360444 % - +36.0%/year compounded
balanced: cagr=0.20735 % - +20.7%/year compounded
macro: cagr=0.42775 % - +42.8%/year compounded


In [22]:
for label, p in portfolios.items():
    a = p.metrics.average_return(geometric=False)
    g = p.metrics.average_return(geometric=True)
    print(f"{label}: average_return arithmetic={a:+.4%}, geometric={g:+.4%}")


growth: average_return arithmetic=+0.0955%, geometric=+0.0844%
balanced: average_return arithmetic=+0.0553%, geometric=+0.0516%
macro: average_return arithmetic=+0.1266%, geometric=+0.0976%


In [23]:
for label, p in portfolios.items():
    r = p.returns()
    excess_vs_rate = excess_returns(r, 0.0001)
    excess_vs_bench = excess_returns(r, benchmark_returns)
    active = active_returns(r, benchmark_returns)
    print(
        f"{label}: excess_returns vs flat rate mean={excess_vs_rate.mean():+.5f}, "
        f"excess_returns vs SPY mean={excess_vs_bench.mean():+.5f}, "
        f"active_returns == excess_returns vs benchmark: {active.equals(excess_vs_bench)}"
    )


growth: excess_returns vs flat rate mean=+0.00085, excess_returns vs SPY mean=+0.00045, active_returns == excess_returns vs benchmark: True
balanced: excess_returns vs flat rate mean=+0.00045, excess_returns vs SPY mean=+0.00005, active_returns == excess_returns vs benchmark: True
macro: excess_returns vs flat rate mean=+0.00117, excess_returns vs SPY mean=+0.00077, active_returns == excess_returns vs benchmark: True


## 7. Metrics -- Risk (`portpy.metrics.risk`)


In [24]:
for label, p in portfolios.items():
    var = p.metrics.variance()
    vol_ann = p.metrics.volatility(annualized=True)
    vol_raw = p.metrics.volatility(annualized=False)
    print(f"{label}: variance={var:.6f}, volatility(annualized)={vol_ann:.2%}, volatility(raw)={vol_raw:.4%}")


growth: variance=0.000221, volatility(annualized)=28.38%, volatility(raw)=1.4857%


balanced: variance=0.000074, volatility(annualized)=16.42%, volatility(raw)=0.8593%
macro: variance=0.000719, volatility(annualized)=51.23%, volatility(raw)=2.6816%


In [25]:
for label, p in portfolios.items():
    dd0 = p.metrics.downside_deviation(mar=0.0)
    dd_mar = p.metrics.downside_deviation(mar=0.02)
    sv = p.metrics.semi_variance(mar=0.0)
    print(f"{label}: downside_deviation(mar=0)={dd0:.2%}, downside_deviation(mar=2%)={dd_mar:.2%}, semi_variance={sv:.6f}")


growth: downside_deviation(mar=0)=19.41%, downside_deviation(mar=2%)=19.45%, semi_variance=0.000103

balanced: downside_deviation(mar=0)=10.63%, downside_deviation(mar=2%)=10.68%, semi_variance=0.000031
macro: downside_deviation(mar=0)=22.19%, downside_deviation(mar=2%)=22.24%, semi_variance=0.000135


In [26]:
for label, p in portfolios.items():
    for method in ("historical", "parametric", "cornish_fisher"):
        for confidence in (0.95, 0.99):
            var = p.metrics.value_at_risk(method=method, confidence=confidence)
            print(f"{label}: VaR[{method}, {confidence:.0%}] = {var:+.2%}")
    cvar = p.metrics.conditional_var(confidence=0.95)
    print(f"{label}: CVaR[95%] = {cvar:+.2%}\n")


growth: VaR[historical, 95%] = -2.08%
growth: VaR[historical, 99%] = -3.69%
growth: VaR[parametric, 95%] = -2.35%
growth: VaR[parametric, 99%] = -3.36%
growth: VaR[cornish_fisher, 95%] = -1.91%
growth: VaR[cornish_fisher, 99%] = -11.63%
growth: CVaR[95%] = -3.27%

balanced: VaR[historical, 95%] = -1.23%
balanced: VaR[historical, 99%] = -2.15%
balanced: VaR[parametric, 95%] = -1.36%
balanced: VaR[parametric, 99%] = -1.94%
balanced: VaR[cornish_fisher, 95%] = -0.76%
balanced: VaR[cornish_fisher, 99%] = -4.34%
balanced: CVaR[95%] = -1.86%

macro: VaR[historical, 95%] = -2.57%
macro: VaR[historical, 99%] = -4.21%
macro: VaR[parametric, 95%] = -4.28%
macro: VaR[parametric, 99%] = -6.11%
macro: VaR[cornish_fisher, 95%] = +50.14%
macro: VaR[cornish_fisher, 99%] = +10.04%
macro: CVaR[95%] = -3.73%



In [27]:
for label, p in portfolios.items():
    print(f"{label}: tail_ratio={p.metrics.tail_ratio():.2f}, skewness={p.metrics.skewness():+.2f}, kurtosis={p.metrics.kurtosis():+.2f}")


growth: tail_ratio=1.16, skewness=-0.59, kurtosis=+22.59
balanced: tail_ratio=1.07, skewness=+1.12, kurtosis=+17.53
macro: tail_ratio=1.06, skewness=+17.06, kurtosis=+497.02


In [28]:
for label, p in portfolios.items():
    ui = p.metrics.ulcer_index()
    pi = p.metrics.pain_index()
    print(f"{label}: ulcer_index={ui:.2%}, pain_index={pi:.2%}")


growth: ulcer_index=10.67%, pain_index=7.80%
balanced: ulcer_index=4.28%, pain_index=3.27%
macro: ulcer_index=16.98%, pain_index=11.38%


In [29]:
for label, p in portfolios.items():
    b = p.metrics.beta(benchmark=benchmark_returns)
    te = p.metrics.tracking_error(benchmark=benchmark_returns)
    print(f"{label}: beta vs SPY={b:+.2f}, tracking_error vs SPY={te:.2%}")


growth: beta vs SPY=+1.31, tracking_error vs SPY=19.93%


balanced: beta vs SPY=+0.56, tracking_error vs SPY=15.48%
macro: beta vs SPY=+0.89, tracking_error vs SPY=49.26%


## 8. Metrics -- Performance (`portpy.metrics.performance`)


In [30]:
for label, p in portfolios.items():
    sharpe = p.metrics.sharpe_ratio(as_result=True)
    sortino = p.metrics.sortino_ratio(mar=0.02, as_result=True)
    print(f"{label}: {sharpe!r}  |  {sortino!r}")


growth: sharpe_ratio=1.08727 - good  |  sortino_ratio=1.69012 - good
balanced: sharpe_ratio=0.986602 - acceptable  |  sortino_ratio=1.70499 - good
macro: sharpe_ratio=0.823758 - acceptable  |  sortino_ratio=1.98758 - good


In [31]:
for label, p in portfolios.items():
    print(f"{label}: {p.metrics.calmar_ratio(as_result=True)!r}")


growth: calmar_ratio=1.25175 - solid
balanced: calmar_ratio=1.30891 - solid
macro: calmar_ratio=0.937782 - weak


In [32]:
for label, p in portfolios.items():
    o0 = p.metrics.omega_ratio(threshold=0.0)
    o_pos = p.metrics.omega_ratio(threshold=0.05)
    print(f"{label}: omega_ratio(threshold=0)={o0:.2f}, omega_ratio(threshold=5%)={o_pos:.2f}")


growth: omega_ratio(threshold=0)=1.22, omega_ratio(threshold=5%)=1.19
balanced: omega_ratio(threshold=0)=1.22, omega_ratio(threshold=5%)=1.16
macro: omega_ratio(threshold=0)=1.22, omega_ratio(threshold=5%)=1.20


In [33]:
for label, p in portfolios.items():
    ir = p.metrics.information_ratio(benchmark=benchmark_returns, as_result=True)
    print(f"{label}: {ir!r}")


growth: information_ratio=0.832569 - good


balanced: information_ratio=0.124249 - weak
macro: information_ratio=0.566957 - good


In [34]:
for label, p in portfolios.items():
    b = p.metrics.beta(benchmark=benchmark_returns)
    tr = p.metrics.treynor_ratio(beta=b, as_result=True)
    print(f"{label}: beta={b:+.2f}  ->  {tr!r}")


growth: beta=+1.31  ->  treynor_ratio=0.235449 - +0.235 excess return per beta unit
balanced: beta=+0.56  ->  treynor_ratio=0.289586 - +0.290 excess return per beta unit
macro: beta=+0.89  ->  treynor_ratio=0.472761 - +0.473 excess return per beta unit


In [35]:
for label, p in portfolios.items():
    m2 = p.metrics.m2_measure(benchmark=benchmark_returns, as_result=True)
    print(f"{label}: {m2!r}")


growth: m2_measure=0.21331 % - +21.3%/yr risk-adjusted return
balanced: m2_measure=0.197328 % - +19.7%/yr risk-adjusted return
macro: m2_measure=0.171476 % - +17.1%/yr risk-adjusted return


In [36]:
for label, p in portfolios.items():
    sterling = p.metrics.sterling_ratio(n=5)
    burke = p.metrics.burke_ratio(n=5)
    print(f"{label}: sterling_ratio(n=5)={sterling:.2f}, burke_ratio(n=5)={burke:.2f}")


growth: sterling_ratio(n=5)=1.60, burke_ratio(n=5)=0.61


balanced: sterling_ratio(n=5)=1.87, burke_ratio(n=5)=0.65


macro: sterling_ratio(n=5)=1.74, burke_ratio(n=5)=0.63


In [37]:
for label, p in portfolios.items():
    print(f"{label}: gain_to_pain_ratio={p.metrics.gain_to_pain_ratio():.2f}")


growth: gain_to_pain_ratio=0.22
balanced: gain_to_pain_ratio=0.22
macro: gain_to_pain_ratio=0.22


In [38]:
for label, p in portfolios.items():
    k3 = p.metrics.kappa_three_ratio(mar=0.0)
    upr = p.metrics.upside_potential_ratio(mar=0.0)
    print(f"{label}: kappa_three_ratio={k3:.2f}, upside_potential_ratio={upr:.2f}")


growth: kappa_three_ratio=0.05, upside_potential_ratio=0.52


balanced: kappa_three_ratio=0.07, upside_potential_ratio=0.56
macro: kappa_three_ratio=0.07, upside_potential_ratio=0.60


## 9. Metrics -- Drawdowns (`portpy.metrics.drawdowns`)


In [39]:
for label, p in portfolios.items():
    dd = p.metrics.drawdown_series()
    mdd = p.metrics.max_drawdown(as_result=True)
    print(f"{label}: drawdown_series range [{dd.min():.2%}, {dd.max():.2%}]  |  {mdd!r}")


growth: drawdown_series range [-28.80%, 0.00%]  |  max_drawdown=-0.287953 % - severe


balanced: drawdown_series range [-15.84%, 0.00%]  |  max_drawdown=-0.158414 % - moderate
macro: drawdown_series range [-45.61%, 0.00%]  |  max_drawdown=-0.45613 % - extreme


In [40]:
for label, p in portfolios.items():
    dur = p.metrics.drawdown_duration()
    ttr = p.metrics.time_to_recovery()
    print(f"{label}: current drawdown_duration={int(dur.iloc[-1])} periods, time_to_recovery(worst episode)={ttr}")


growth: current drawdown_duration=321 periods, time_to_recovery(worst episode)=None


balanced: current drawdown_duration=21 periods, time_to_recovery(worst episode)=129.0


macro: current drawdown_duration=343 periods, time_to_recovery(worst episode)=None


In [41]:
for label, p in portfolios.items():
    print(f"--- {label} top_n_drawdowns(n=3) ---")
    print(p.metrics.top_n_drawdowns(n=3))


--- growth top_n_drawdowns(n=3) ---
       start     trough        end   depth  duration_to_trough  duration_to_recovery  recovered
0 2025-10-28 2026-03-30        NaT -0.2880                 153                   NaN      False
1 2024-12-16 2025-04-08 2025-06-25 -0.2843                 113              191.0000       True
2 2024-06-05 2024-08-07 2024-12-11 -0.2774                  63              189.0000       True
--- balanced top_n_drawdowns(n=3) ---
       start     trough        end   depth  duration_to_trough  duration_to_recovery  recovered
0 2025-01-31 2025-04-08 2025-06-09 -0.1584                  67              129.0000       True
1 2026-01-28 2026-02-05 2026-05-01 -0.1088                   8               93.0000       True
2 2026-05-11 2026-06-30 2026-08-20 -0.1038                  50              101.0000       True
--- macro top_n_drawdowns(n=3) ---
       start     trough        end   depth  duration_to_trough  duration_to_recovery  recovered
0 2025-10-06 2026-06-10    

In [42]:
for label, p in portfolios.items():
    avg_dd = p.metrics.average_drawdown(as_result=True)
    dar_95 = p.metrics.drawdown_at_risk(confidence=0.95)
    dar_99 = p.metrics.drawdown_at_risk(confidence=0.99)
    print(f"{label}: {avg_dd!r}  |  drawdown_at_risk[95%]={dar_95:.2%}, [99%]={dar_99:.2%}")


growth: average_drawdown=-0.0333797 % - -3.34% average drawdown depth  |  drawdown_at_risk[95%]=-21.41%, [99%]=-25.32%


balanced: average_drawdown=-0.0240897 % - -2.41% average drawdown depth  |  drawdown_at_risk[95%]=-8.33%, [99%]=-10.36%


macro: average_drawdown=-0.0483221 % - -4.83% average drawdown depth  |  drawdown_at_risk[95%]=-39.90%, [99%]=-43.35%


In [43]:
for label, p in portfolios.items():
    cdar_95 = p.metrics.conditional_drawdown_at_risk(confidence=0.95)
    cdar_99 = p.metrics.conditional_drawdown_at_risk(confidence=0.99)
    print(f"{label}: CDaR[95%]={cdar_95:.2%}, CDaR[99%]={cdar_99:.2%}")

growth: CDaR[95%]=-23.92%, CDaR[99%]=-27.11%
balanced: CDaR[95%]=-9.71%, CDaR[99%]=-12.01%
macro: CDaR[95%]=-42.11%, CDaR[99%]=-44.37%


## 10. Metrics -- Rolling & Expanding Windows (`portpy.metrics.rolling`)


In [44]:
from portpy.metrics.risk import skewness as _skewness
from portpy.metrics.rolling import rolling_metric

WINDOW = 60
for label, p in portfolios.items():
    r = p.returns()
    roll_skew = rolling_metric(r, _skewness, window=WINDOW)
    roll_sharpe = p.metrics.rolling_sharpe(window=WINDOW)
    roll_vol = p.metrics.rolling_volatility(window=WINDOW)
    print(
        f"{label}: rolling_metric(skewness, w={WINDOW}) last={roll_skew.dropna().iloc[-1]:+.2f}, "
        f"rolling_sharpe last={roll_sharpe.dropna().iloc[-1]:+.2f}, "
        f"rolling_volatility last={roll_vol.dropna().iloc[-1]:.2%}"
    )


growth: rolling_metric(skewness, w=60) last=+1.07, rolling_sharpe last=+2.67, rolling_volatility last=22.88%


balanced: rolling_metric(skewness, w=60) last=+1.63, rolling_sharpe last=+4.10, rolling_volatility last=13.38%


macro: rolling_metric(skewness, w=60) last=+1.45, rolling_sharpe last=+3.14, rolling_volatility last=31.33%


In [45]:
from portpy.metrics.rolling import rolling_correlation

for label, p in portfolios.items():
    rb = p.metrics.rolling_beta(benchmark=benchmark_returns, window=WINDOW)
    print(f"{label}: rolling_beta(w={WINDOW}) last={rb.dropna().iloc[-1]:+.2f}")

rc = rolling_correlation(p_growth.returns(), p_macro.returns(), window=WINDOW)
print(f"rolling_correlation(growth vs macro, w={WINDOW}) last={rc.dropna().iloc[-1]:+.2f}")


growth: rolling_beta(w=60) last=+1.44
balanced: rolling_beta(w=60) last=+0.29
macro: rolling_beta(w=60) last=+1.24
rolling_correlation(growth vs macro, w=60) last=+0.84


In [46]:
from portpy.metrics.performance import sharpe_ratio as _sharpe_ratio
from portpy.metrics.rolling import expanding_metric

for label, p in portfolios.items():
    exp_sharpe = expanding_metric(p.returns(), _sharpe_ratio, min_periods=30)
    print(f"{label}: expanding_metric(sharpe_ratio) last value={exp_sharpe.dropna().iloc[-1]:+.2f}")


growth: expanding_metric(sharpe_ratio) last value=+1.02


balanced: expanding_metric(sharpe_ratio) last value=+1.02
macro: expanding_metric(sharpe_ratio) last value=+0.75


## 11. Metrics -- Distributions (`portpy.metrics.distributions`)


In [47]:
for label, p in portfolios.items():
    print(f"--- {label} describe() ---")
    print(p.metrics.describe())


--- growth describe() ---
count      1,459.0000
mean           0.0010
std            0.0149
min           -0.1831
25%           -0.0052
50%            0.0005
75%            0.0072
max            0.1309
skew          -0.5921
kurtosis      22.5938
dtype: float64
--- balanced describe() ---
count      1,459.0000
mean           0.0006
std            0.0086
min           -0.0512
25%           -0.0033
50%            0.0004
75%            0.0043
max            0.1031
skew           1.1195
kurtosis      17.5265
dtype: float64
--- macro describe() ---
count      1,459.0000
mean           0.0013
std            0.0268
min           -0.1221
25%           -0.0080
50%            0.0003
75%            0.0095
max            0.7835
skew          17.0590
kurtosis     497.0250
dtype: float64


In [48]:
for label, p in portfolios.items():
    nt = p.metrics.normality_test()
    print(f"{label}: {nt}")


growth: {'statistic': 30894.50240837186, 'p_value': 0.0, 'is_normal': False, '_portpy_explain_name': 'normality_test'}
balanced: {'statistic': 18841.536376146883, 'p_value': 0.0, 'is_normal': False, '_portpy_explain_name': 'normality_test'}
macro: {'statistic': 14985312.771965094, 'p_value': 0.0, 'is_normal': False, '_portpy_explain_name': 'normality_test'}


In [49]:
for label, p in portfolios.items():
    bw = p.metrics.best_worst_periods(n=3)
    print(f"--- {label} ---")
    print("best:")
    print(bw["best"])
    print("worst:")
    print(bw["worst"])


--- growth ---
best:
timestamp
2025-04-09   0.1309
2022-11-10   0.1091
2023-05-25   0.0646
Name: Growth & Crypto Long/Short, dtype: float64
worst:
timestamp
2024-06-10   -0.1831
2022-11-09   -0.0573
2024-08-05   -0.0530
Name: Growth & Crypto Long/Short, dtype: float64
--- balanced ---
best:
timestamp
2024-06-10   0.1031
2025-04-09   0.0550
2022-11-10   0.0408
Name: Diversified Multi-Asset Balanced, dtype: float64
worst:
timestamp
2026-02-05   -0.0512
2026-01-30   -0.0430
2022-11-09   -0.0374
Name: Diversified Multi-Asset Balanced, dtype: float64
--- macro ---
best:
timestamp
2024-08-26   0.7835
2022-11-10   0.1095
2025-03-02   0.0811
Name: Long/Short Macro, dtype: float64
worst:
timestamp
2022-11-09   -0.1221
2026-02-05   -0.0940
2025-03-03   -0.0735
Name: Long/Short Macro, dtype: float64


In [50]:
for label, p in portfolios.items():
    wr = p.metrics.win_rate()
    wlr = p.metrics.win_loss_ratio()
    ppp = p.metrics.positive_periods_pct()
    print(f"{label}: win_rate={wr:.1%}, win_loss_ratio={wlr:.2f}, positive_periods_pct={ppp:.1%} (matches win_rate: {np.isclose(wr, ppp)})")


growth: win_rate=53.7%, win_loss_ratio=1.05, positive_periods_pct=53.7% (matches win_rate: True)
balanced: win_rate=54.2%, win_loss_ratio=1.03, positive_periods_pct=54.2% (matches win_rate: True)
macro: win_rate=51.8%, win_loss_ratio=1.14, positive_periods_pct=51.8% (matches win_rate: True)


In [51]:
for label, p in portfolios.items():
    print(f"--- {label} monthly_returns_table() (tail) ---")
    print(p.metrics.monthly_returns_table().tail(4))


--- growth monthly_returns_table() (tail) ---
         Jan     Feb     Mar     Apr    May     Jun     Jul     Aug     Sep    Oct     Nov     Dec   Year
year                                                                                                     
2023  0.1981  0.0508  0.1715  0.0173 0.1200  0.0755  0.0185 -0.0402 -0.0654 0.0721  0.1318  0.0567 1.1170
2024  0.0456  0.1648  0.0474 -0.0676 0.1527 -0.1352 -0.0094 -0.0160  0.0402 0.0173  0.1274  0.0089 0.3912
2025 -0.0101 -0.0626 -0.0864  0.0538 0.1381  0.0571  0.0906  0.0230  0.0672 0.0290 -0.0755 -0.0081 0.2076
2026 -0.0721 -0.0732 -0.0479  0.1298 0.0751 -0.1148  0.0638  0.1214 -0.0049    NaN     NaN     NaN 0.0451
--- balanced monthly_returns_table() (tail) ---
         Jan     Feb     Mar     Apr     May     Jun    Jul     Aug     Sep     Oct     Nov     Dec   Year
year                                                                                                      
2023  0.0645 -0.0414  0.0279  0.0229 -0.0598  0.0336 0.0

In [52]:
for label, p in portfolios.items():
    counts, edges = p.metrics.return_histogram_data(bins=20)
    print(f"{label}: return_histogram_data -> {len(counts)} bins, busiest bin count={counts.max()}, range=[{edges[0]:+.3%}, {edges[-1]:+.3%}]")


growth: return_histogram_data -> 20 bins, busiest bin count=817, range=[-18.305%, +13.094%]
balanced: return_histogram_data -> 20 bins, busiest bin count=701, range=[-5.119%, +10.310%]
macro: return_histogram_data -> 20 bins, busiest bin count=1176, range=[-12.213%, +78.347%]


## 12. Metrics -- Benchmark-Relative (`portpy.metrics.benchmarks`)


In [53]:
for label, p in portfolios.items():
    a = p.metrics.alpha(benchmark=benchmark_returns, as_result=True)
    corr = p.metrics.correlation(benchmark=benchmark_returns)
    r2 = p.metrics.r_squared(benchmark=benchmark_returns)
    print(f"{label}: {a!r}  |  correlation={corr:+.2f}  |  r_squared={r2:.1%}")


growth: alpha=0.129206 % - +12.9%/yr (positive excess performance)  |  correlation=+0.73  |  r_squared=53.7%
balanced: alpha=0.0855927 % - +8.6%/yr (positive excess performance)  |  correlation=+0.54  |  r_squared=29.3%
macro: alpha=0.342443 % - +34.2%/yr (positive excess performance)  |  correlation=+0.28  |  r_squared=7.7%


In [54]:
for label, p in portfolios.items():
    up = p.metrics.up_capture_ratio(benchmark=benchmark_returns)
    down = p.metrics.down_capture_ratio(benchmark=benchmark_returns)
    overall = p.metrics.capture_ratio(benchmark=benchmark_returns)
    print(f"{label}: up_capture={up:.2f}, down_capture={down:.2f}, capture_ratio={overall:.2f}")


growth: up_capture=4.04, down_capture=1.03, capture_ratio=1.94
balanced: up_capture=0.20, down_capture=0.84, capture_ratio=1.12
macro: up_capture=0.87, down_capture=0.96, capture_ratio=2.31


In [55]:
for label, p in portfolios.items():
    ba = p.metrics.batting_average(benchmark=benchmark_returns)
    print(f"{label}: batting_average={ba:.1%}")


growth: batting_average=53.1%
balanced: batting_average=50.3%
macro: batting_average=49.4%


## 13. Metrics -- Covariance & Risk Decomposition (`portpy.metrics.covariance`)


In [56]:
for label, p in portfolios.items():
    corr = p.metrics.correlation_matrix()
    print(f"--- {label} correlation_matrix ---")
    print(corr.round(2))


--- growth correlation_matrix ---
       AAPL   MSFT   NVDA    QQQ    BTC    ETH     GLD    CASH     XOM
AAPL 1.0000 0.4400 0.3000 0.6300 0.1800 0.2100  0.0700  0.0100  0.1700
MSFT 0.4400 1.0000 0.3500 0.6600 0.2300 0.2400  0.0900  0.0300  0.0300
NVDA 0.3000 0.3500 1.0000 0.5400 0.1700 0.2000  0.0700  0.0100  0.0100
QQQ  0.6300 0.6600 0.5400 1.0000 0.3300 0.3600  0.2000  0.0200  0.1000
BTC  0.1800 0.2300 0.1700 0.3300 1.0000 0.8300  0.1500  0.0400  0.0600
ETH  0.2100 0.2400 0.2000 0.3600 0.8300 1.0000  0.1300  0.0200  0.0700
GLD  0.0700 0.0900 0.0700 0.2000 0.1500 0.1300  1.0000 -0.0100  0.0600
CASH 0.0100 0.0300 0.0100 0.0200 0.0400 0.0200 -0.0100  1.0000 -0.0300
XOM  0.1700 0.0300 0.0100 0.1000 0.0600 0.0700  0.0600 -0.0300  1.0000
--- balanced correlation_matrix ---
        SPY    IWM    JPM     GLD     SLV     USO     BTC    ETH    CASH   NVDA
SPY  1.0000 0.8300 0.6000  0.1900  0.2900  0.0600  0.3300 0.3700  0.0200 0.4800
IWM  0.8300 1.0000 0.5900  0.2000  0.2800  0.0400  0.3600 0.

In [57]:
for label, p in portfolios.items():
    pv = p.metrics.portfolio_variance(as_result=True)
    pvol = p.metrics.portfolio_volatility(as_result=True)
    print(f"{label}: {pv!r}  |  {pvol!r}")


growth: portfolio_variance=0.000220737  |  portfolio_volatility=0.0148572 - 1.49%
balanced: portfolio_variance=7.38322e-05  |  portfolio_volatility=0.00859256 - 0.86%
macro: portfolio_variance=0.000719072  |  portfolio_volatility=0.0268155 - 2.68%


In [58]:
for label, p in portfolios.items():
    dr = p.metrics.diversification_ratio(as_result=True)
    print(f"{label}: {dr!r}")


growth: diversification_ratio=1.3726 - 1.37x diversification benefit
balanced: diversification_ratio=1.26539 - 1.27x diversification benefit
macro: diversification_ratio=1.35736 - 1.36x diversification benefit


In [59]:
for label, p in portfolios.items():
    mctr = p.metrics.marginal_contribution_to_risk()
    cctr = p.metrics.component_contribution_to_risk()
    breakdown = pd.DataFrame({"weight": p.weights, "MCTR": mctr, "CCTR": cctr, "CCTR_pct": cctr / cctr.sum()})
    print(f"--- {label} risk decomposition ---")
    print(breakdown.round(4))


--- growth risk decomposition ---
      weight    MCTR   CCTR  CCTR_pct
AAPL  0.2444  0.0084 0.0021    0.1384
MSFT  0.2000  0.0091 0.0018    0.1224
NVDA  0.2000  0.0261 0.0052    0.3518
QQQ   0.1333  0.0086 0.0011    0.0774
BTC   0.1556  0.0154 0.0024    0.1613
ETH   0.0889  0.0215 0.0019    0.1287
GLD   0.0889  0.0021 0.0002    0.0128
CASH  0.0556  0.0000 0.0000    0.0000
XOM  -0.1667 -0.0006 0.0001    0.0072
--- balanced risk decomposition ---
      weight    MCTR   CCTR  CCTR_pct
SPY   0.2174  0.0045 0.0010    0.1137
IWM   0.1087  0.0068 0.0007    0.0859
JPM   0.1304  0.0061 0.0008    0.0925
GLD   0.1304  0.0049 0.0006    0.0743
SLV   0.0870  0.0110 0.0010    0.1117
USO   0.0870  0.0045 0.0004    0.0457
BTC   0.1087  0.0174 0.0019    0.2195
ETH   0.0761  0.0238 0.0018    0.2112
CASH  0.1630  0.0000 0.0000    0.0000
NVDA -0.1087 -0.0036 0.0004    0.0455
--- macro risk decomposition ---
      weight    MCTR    CCTR  CCTR_pct
MSFT  0.1176  0.0032  0.0004    0.0139
QQQ   0.1765  0.0030 

In [60]:
for label, p in portfolios.items():
    dr_annualized_cov = p.metrics.diversification_ratio(cov_matrix=p.metrics.covariance_matrix(annualized=True))
    print(f"{label}: diversification_ratio using an explicit annualized cov_matrix override = {dr_annualized_cov:.2f}")


growth: diversification_ratio using an explicit annualized cov_matrix override = 1.37
balanced: diversification_ratio using an explicit annualized cov_matrix override = 1.27
macro: diversification_ratio using an explicit annualized cov_matrix override = 1.36


## 14. Metrics -- One-Shot Summaries (`portpy.metrics.summary`)


In [61]:
for label, p in portfolios.items():
    print("=" * 20, label.upper(), "TEARSHEET", "=" * 20)
    for metric_name, result in p.metrics.tearsheet_summary().items():
        print(f"  {metric_name:22s} {float(result):>10.4f}   ({result.interpretation})")
    print()


==================== GROWTH TEARSHEET ====================
  total_return               2.4226   (+242.3% total return)
  annualized_return          0.3604   (+36.0% annualized return)
  cagr                       0.3604   (+36.0%/year compounded)
  volatility                 0.2838   (28.4%/yr (high - equity-like or more volatile))
  sharpe_ratio               1.0873   (good)
  sortino_ratio              1.7960   (good)
  calmar_ratio               1.2517   (solid)
  max_drawdown              -0.2880   (severe)
  value_at_risk_95          -0.0208   (-2.08% - expect a worse-than-this loss only in the excluded tail probability)
  conditional_var_95        -0.0327   (-3.27% average loss in the worst-case tail)
  skewness                  -0.5921   (-0.59 (negative skew - watch for rare large losses))
  kurtosis                  22.5938   (+22.59 (fat tails - Normal-based risk estimates will understate real risk))
  win_rate                   0.5374   (53.7% of periods were positive)

===

In [62]:
for label, p in portfolios.items():
    print(f"--- {label} vs SPY ---")
    print(p.metrics.compare_to_benchmark(benchmark=benchmark_returns))
    print()


--- growth vs SPY ---
                    portfolio  benchmark  difference
annualized_return      0.3604     0.1854      0.1751
volatility             0.2838     0.1588      0.1251
sharpe_ratio           1.0873     0.8991      0.1882
max_drawdown          -0.2880    -0.1900     -0.0980
beta                   1.3108        NaN         NaN
alpha                  0.1292        NaN         NaN
correlation            0.7331        NaN         NaN
information_ratio      0.8326        NaN         NaN
up_capture_ratio       4.0406        NaN         NaN
down_capture_ratio     1.0266        NaN         NaN
batting_average        0.5312        NaN         NaN

--- balanced vs SPY ---
                    portfolio  benchmark  difference
annualized_return      0.2073     0.1854      0.0220
volatility             0.1642     0.1588      0.0054
sharpe_ratio           0.9866     0.8991      0.0876
max_drawdown          -0.1584    -0.1900      0.0316
beta                   0.5593        NaN         NaN

## 15. Metrics -- Transaction Costs (`portpy.metrics.costs`)


In [63]:
from portpy.metrics.costs import net_of_costs_returns, turnover_from_weights
from portpy.metrics.returns import total_return as _total_return


def simulate_weight_drift(returns_df: pd.DataFrame, target_weights: pd.Series, rebalance_every: int = 21) -> pd.DataFrame:
    # Buy-and-hold weight drift between periodic rebalances back to target - a stand-in for a real trade log.
    aligned = returns_df[target_weights.index]
    history = []
    current = target_weights.copy()
    for i, date in enumerate(aligned.index):
        if i > 0 and i % rebalance_every == 0:
            current = target_weights.copy()
        history.append(current.copy())
        port_r = float((current * aligned.loc[date]).sum())
        current = current * (1.0 + aligned.loc[date]) / (1.0 + port_r)
    return pd.DataFrame(history, index=aligned.index)


for label, p in portfolios.items():
    weights_history = simulate_weight_drift(p.asset_returns(), p.weights, rebalance_every=21)
    turnover = turnover_from_weights(weights_history)
    print(f"{label}: turnover_from_weights -> mean={turnover.mean():.2%}, max={turnover.max():.2%} (rebalance days spike, drift days near 0)")


growth: turnover_from_weights -> mean=0.90%, max=66.67% (rebalance days spike, drift days near 0)


balanced: turnover_from_weights -> mean=0.71%, max=60.87% (rebalance days spike, drift days near 0)


macro: turnover_from_weights -> mean=1.26%, max=79.41% (rebalance days spike, drift days near 0)


In [64]:
for label, p in portfolios.items():
    weights_history = simulate_weight_drift(p.asset_returns(), p.weights, rebalance_every=21)
    print(weights_history.tail(5))
    turnover = turnover_from_weights(weights_history)
    net_variable = net_of_costs_returns(p.returns(), turnover, cost_bps=10)
    net_constant = net_of_costs_returns(p.returns(), 0.05, cost_bps=10)
    gross_total = _total_return(p.returns())
    net_variable_total = _total_return(net_variable)
    net_constant_total = _total_return(net_constant)
    print(
        f"{label}: gross total_return={gross_total:+.2%}, "
        f"net (variable turnover)={net_variable_total:+.2%}, "
        f"net (constant 5% turnover)={net_constant_total:+.2%}"
    )

    portpy.explain("turnover_from_weights")

             AAPL   MSFT   NVDA    QQQ    BTC    ETH    GLD   CASH     XOM
timestamp                                                                 
2026-09-10 0.2461 0.2010 0.1984 0.1357 0.1561 0.0912 0.0900 0.0568 -0.1753
2026-09-11 0.2560 0.2022 0.1945 0.1348 0.1533 0.0905 0.0889 0.0570 -0.1772
2026-09-12 0.2576 0.2013 0.1923 0.1345 0.1530 0.0924 0.0884 0.0564 -0.1761
2026-09-13 0.2575 0.2012 0.1923 0.1345 0.1530 0.0927 0.0884 0.0564 -0.1760
2026-09-14 0.2582 0.2018 0.1928 0.1348 0.1525 0.0912 0.0886 0.0566 -0.1765
growth: gross total_return=+242.26%, net (variable turnover)=+237.82%, net (constant 5% turnover)=+218.20%
turnover_from_weights (function)

What it is:
  Measures how much of the portfolio is traded between periods by analyzing changes in portfolio allocation weights.

Formula:
  0.5 * sum(abs(current_weight - previous_weight))

How to read it:
  A turnover value of 0.20 means that 20% of portfolio value was exchanged during that period. The metric reflects portfolio tr

              SPY    IWM    JPM    GLD    SLV    USO    BTC    ETH   CASH    NVDA
timestamp                                                                        
2026-09-10 0.2149 0.1066 0.1288 0.1292 0.0882 0.0918 0.1067 0.0763 0.1629 -0.1054
2026-09-11 0.2148 0.1061 0.1291 0.1276 0.0839 0.0974 0.1048 0.0758 0.1638 -0.1035
2026-09-12 0.2153 0.1059 0.1293 0.1276 0.0843 0.0947 0.1051 0.0778 0.1628 -0.1028
2026-09-13 0.2152 0.1058 0.1292 0.1276 0.0843 0.0947 0.1052 0.0780 0.1628 -0.1028
2026-09-14 0.2157 0.1060 0.1295 0.1278 0.0845 0.0949 0.1047 0.0767 0.1632 -0.1030
balanced: gross total_return=+112.38%, net (variable turnover)=+110.18%, net (constant 5% turnover)=+97.44%
turnover_from_weights (function)

What it is:
  Measures how much of the portfolio is traded between periods by analyzing changes in portfolio allocation weights.

Formula:
  0.5 * sum(abs(current_weight - previous_weight))

How to read it:
  A turnover value of 0.20 means that 20% of portfolio value was exchanged du

             MSFT    QQQ    BTC    ETH    SOL    GLD     USO     SPY   CASH
timestamp                                                                  
2026-09-10 0.1173 0.1782 0.2929 0.1797 0.1188 0.1774 -0.1260 -0.1771 0.2387
2026-09-11 0.1203 0.1805 0.2931 0.1816 0.1181 0.1784 -0.1362 -0.1802 0.2443
2026-09-12 0.1189 0.1788 0.2905 0.1842 0.1205 0.1763 -0.1308 -0.1785 0.2400
2026-09-13 0.1189 0.1788 0.2907 0.1849 0.1197 0.1763 -0.1308 -0.1785 0.2400
2026-09-14 0.1199 0.1803 0.2913 0.1828 0.1177 0.1778 -0.1319 -0.1800 0.2420
macro: gross total_return=+315.13%, net (variable turnover)=+307.56%, net (constant 5% turnover)=+285.95%
turnover_from_weights (function)

What it is:
  Measures how much of the portfolio is traded between periods by analyzing changes in portfolio allocation weights.

Formula:
  0.5 * sum(abs(current_weight - previous_weight))

How to read it:
  A turnover value of 0.20 means that 20% of portfolio value was exchanged during that period. The metric reflects portfo

### 15.1 Itemized Costs: `bid_ask_spreads`, `broker_commissions`, `fx_spread_costs`, `tax_impact`

The blended `cost_bps` above is a single flat-rate approximation. These four functions itemize the same drag by source instead - each returns a per-period return-drag Series in the same units, so they compose by plain addition before subtracting from returns.


In [65]:
from portpy.metrics.costs import bid_ask_spreads, broker_commissions, fx_spread_costs, tax_impact

for label, p in portfolios.items():
    weights_history = simulate_weight_drift(p.asset_returns(), p.weights, rebalance_every=21)
    turnover = turnover_from_weights(weights_history)

    spread_cost = bid_ask_spreads(weights_history, spread_bps=5.0)
    commission_cost = broker_commissions(weights_history, commission_bps=2.0)
    # Treat JPM as a EUR-denominated holding for this illustration, everything else already in USD.
    asset_currencies = {a: "EUR" for a in weights_history.columns if a == "JPM"}
    fx_cost = fx_spread_costs(weights_history, asset_currencies, base_currency="USD", fx_spread_bps=3.0)
    tax_adjusted = tax_impact(
        p.returns(), turnover, capital_gains_tax_rate=0.15, dividend_tax_rate=0.15, dividend_yield=0.0001
    )

    total_drag = spread_cost + commission_cost + fx_cost
    gross_total = _total_return(p.returns())
    itemized_net_total = _total_return(p.returns() - total_drag)
    tax_adjusted_total = _total_return(tax_adjusted)
    print(
        f"{label}: gross={gross_total:+.2%}, "
        f"net of itemized trading costs (spread+commission+fx)={itemized_net_total:+.2%}, "
        f"net of estimated tax drag={tax_adjusted_total:+.2%}"
    )

portpy.explain("bid_ask_spreads")
portpy.explain("fx_spread_costs")
portpy.explain("tax_impact")

growth: gross=+242.26%, net of itemized trading costs (spread+commission+fx)=+238.26%, net of estimated tax drag=+231.43%


balanced: gross=+112.38%, net of itemized trading costs (spread+commission+fx)=+110.35%, net of estimated tax drag=+106.82%


macro: gross=+315.13%, net of itemized trading costs (spread+commission+fx)=+308.31%, net of estimated tax drag=+298.76%
bid_ask_spreads (function)

What it is:
  Estimates the cost of crossing the bid-ask spread on every trade, from an explicit per-asset weight history.

Formula:
  sum_i(|delta w_i,t| * spread_bps_i / 10000 / 2), first period charges the full initial allocation

How to read it:
  A per-period cost drag in return units, same convention as net_of_costs_returns' cost_drag - subtract it from returns directly, or add it to other itemized cost Series first.

Good vs. bad:
  Lower spread cost for the same amount of rebalancing activity means either tighter-spread (typically larger, more liquid) assets or per-asset spread assumptions that may be too optimistic - check the spread_bps you supplied against real quoted spreads for the actual instruments.

Caveats:
  Uses quoted top-of-book spread and a flat half-spread-per-leg convention - it says nothing about market impact from

"tax_impact (function)\n=====================\n\nWhat it is:\n  Estimates a tax drag on returns from realized capital gains (proxied by turnover) plus a dividend tax on an assumed dividend yield.\n\nFormula:\n  returns - clip(returns, min=0)*turnover*capital_gains_tax_rate - dividend_yield*dividend_tax_rate\n\nHow to read it:\n  A widening gap between gross and tax-adjusted returns over a high-turnover period means the estimated realized-gains tax is doing real work - compare against net_of_costs_returns' transaction-cost drag to see which friction dominates.\n\nGood vs. bad:\n  Lower estimated tax drag for the same investment thesis (e.g. via lower turnover, tax-advantaged account assumptions, or long-term vs. short-term rate choice) is generically 'better' after-tax, all else equal - but don't tune capital_gains_tax_rate downward just to make a backtest look better than your actual tax situation.\n\nCaveats:\n  A first-order, single-effective-rate approximation, not a lot-level tax e

## 16. The Explainability Layer (`portpy.explain`)


In [66]:
print("Registered explanation names by category:")
print(" metric:", len(available("metric")))
print(" chart: ", len(available("chart")))

portpy.explain("sharpe_ratio")


Registered explanation names by category:
 metric: 63
 chart:  0
sharpe_ratio (metric)

What it is:
  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.

Formula:
  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)

How to read it:
  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.

Good vs. bad:
  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.

Caveats:
  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests. The sqrt(periods_per_year) annualization assumes i.i.d., serially uncorrelated returns - positive autocorrelation (illiquid/infrequently-priced assets) means the annualize

'sharpe_ratio (metric)\n=====================\n\nWhat it is:\n  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.\n\nFormula:\n  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)\n\nHow to read it:\n  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.\n\nGood vs. bad:\n  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.\n\nCaveats:\n  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests. The sqrt(periods_per_year) annualization assumes i.i.d., serially uncorrelated returns - positive autocorrelation (illiquid/infrequently-priced assets) means the annualized figure is overstated (Lo

In [67]:
sharpe_result = p_growth.metrics.sharpe_ratio(as_result=True)
print("isinstance of float:", isinstance(sharpe_result, float))
print("str():", str(sharpe_result))
print("repr():", repr(sharpe_result))
print("arithmetic still works: sharpe_result * 2 =", sharpe_result * 2)
print("interpretation:", sharpe_result.interpretation)
sharpe_result.explain()


isinstance of float: True
str(): 1.0872749761033949
repr(): sharpe_ratio=1.08727 - good
arithmetic still works: sharpe_result * 2 = 2.1745499522067897
interpretation: good
sharpe_ratio (metric)

What it is:
  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.

Formula:
  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)

How to read it:
  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.

Good vs. bad:
  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.

Caveats:
  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests. The sqrt(periods_per_year) annualization assumes i.i.d., seriall

'sharpe_ratio (metric)\n=====================\n\nWhat it is:\n  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.\n\nFormula:\n  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)\n\nHow to read it:\n  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.\n\nGood vs. bad:\n  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.\n\nCaveats:\n  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests. The sqrt(periods_per_year) annualization assumes i.i.d., serially uncorrelated returns - positive autocorrelation (illiquid/infrequently-priced assets) means the annualized figure is overstated (Lo

In [68]:
comparison = p_growth.metrics.compare_to_benchmark(benchmark=benchmark_returns)
portpy.explain(comparison)  # dispatches via comparison.attrs["portpy_explanation"]

portpy.explain(sharpe_result)  # dispatches via MetricResult.name, auto-injecting its own value


compare_to_benchmark (metric)

What it is:
  A structured comparison between a portfolio and a benchmark showing absolute performance, risk characteristics, and benchmark-relative statistics.

Formula:
  comparison = portfolio_metrics - benchmark_metrics + relative_metrics

How to read it:
  Positive differences generally indicate portfolio outperformance for return-based metrics. For risk metrics, the preferred direction depends on the objective, such as lower volatility or smaller drawdown.

Good vs. bad:
  A favorable comparison usually combines higher risk-adjusted returns, lower downside risk, positive alpha, strong information ratio, and appropriate benchmark exposure.

Caveats:
  The quality of the comparison depends on benchmark selection. A poorly chosen benchmark can make relative performance conclusions misleading. The "benchmark" column is intentionally NaN for metrics that are inherently relative/one-sided (beta, alpha, correlation, information_ratio, up_capture_ratio, dow

'sharpe_ratio (metric)\n=====================\n\nWhat it is:\n  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.\n\nFormula:\n  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)\n\nHow to read it:\n  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.\n\nGood vs. bad:\n  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.\n\nCaveats:\n  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests. The sqrt(periods_per_year) annualization assumes i.i.d., serially uncorrelated returns - positive autocorrelation (illiquid/infrequently-priced assets) means the annualized figure is overstated (Lo

In [69]:
card = get("max_drawdown")
print(card.name, "|", card.category)
print(card.summary)
print(card.formula)


max_drawdown | metric
The largest observed loss from a historical peak to the following lowest point before a recovery or the end of the sample.
minimum(drawdown_series)


## 17. `Portfolio.metrics` Auto-Fill Mechanics

`portfolio.metrics.<fn>()` is a thin, dynamically-bound wrapper around the plain function -
useful for debugging exactly what it does and doesn't auto-supply for you.


In [70]:
from portpy.metrics.risk import beta as _beta
from portpy.metrics.risk import volatility as _volatility

p = p_balanced

# 1. No args: every eligible parameter (`returns`, `periods_per_year`, ...) is auto-filled from the portfolio.
auto_vol = p.metrics.volatility()
manual_vol = _volatility(p.returns(), periods_per_year=p.frequency)
print("volatility(): auto-fill matches manual call:", np.isclose(auto_vol, manual_vol))

# 2. kwargs still trigger auto-fill for every OTHER param - only the one you name is overridden.
override_vol = p.metrics.volatility(periods_per_year=252)
print(f"volatility(periods_per_year=252) -> {override_vol:.2%}  (portfolio frequency is {p.frequency})")

# 3. `benchmark` isn't one of the auto-filled names (returns/y/prices/weights/cov_matrix/rf/periods_per_year),
#    so it must always be supplied explicitly - auto-fill only ever covers the portfolio's OWN data.
auto_beta = p.metrics.beta(benchmark=benchmark_returns)
manual_beta = _beta(p.returns(), benchmark_returns)
print("beta(benchmark=...): auto-fill matches manual call:", np.isclose(auto_beta, manual_beta))

# 4. Any positional argument disables auto-fill for the WHOLE call - the guard the docstring warns about.
positional_vol = p.metrics.volatility(p.returns(), True)  # (returns, annualized) positionally
print(
    f"volatility(returns, annualized=True) positionally -> {positional_vol:.4%} "
    f"(periods_per_year silently reverts to the function's own default of 252, not p.frequency={p.frequency})"
)


volatility(): auto-fill matches manual call: True
volatility(periods_per_year=252) -> 13.64%  (portfolio frequency is 365)
beta(benchmark=...): auto-fill matches manual call: True
volatility(returns, annualized=True) positionally -> 13.6403% (periods_per_year silently reverts to the function's own default of 252, not p.frequency=365)
